# EDA - Case Sorveteria Analytics

Analise exploratoria inicial da base bruta `data/raw/vendas_sorvetes.csv`.

Escopo:

- identificar tipos de dados;
- verificar valores nulos;
- verificar duplicidades;
- identificar inconsistencias;
- validar formatos;
- gerar estatisticas descritivas;
- identificar padroes iniciais;
- sugerir KPIs executivos relevantes;
- sugerir possiveis insights relacionados a aumento de receita.

Esta etapa nao faz limpeza pesada, nao altera `data/raw` e nao cria visualizacoes complexas.

In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import eda

PROJECT_ROOT

WindowsPath('C:/Users/User/OneDrive/Documentos/David/case-sorveteria-analytics')

## Carregamento Da Base Bruta

In [2]:
df = eda.load_raw_data()
df.shape

(50000, 12)

## Quantidade De Linhas E Colunas

In [3]:
estrutura = eda.summarize_structure(df)
estrutura

{'linhas': 50000,
 'colunas': 12,
 'memoria_mb': 23.2,
 'colunas_esperadas_presentes': True}

## Primeiras Linhas

In [4]:
df.head()

,ID_Transacao,Data,Hora,Tipo_Sorvete,Sabor,Quantidade,Valor_Total,Cidade,Estado,Canal_Venda,Promocao,ID_Cliente
0,1,2025-04-30,11:00,Milkshake,Açaí,3,16.22,Campinas,AP,App,True,CLI9795
1,2,2025-07-12,15:45,Milkshake,Menta,4,18.57,Aragão,TO,App,True,CLI1799
2,3,2025-03-08,15:15,Pote,Baunilha,4,21.76,Câmara,PA,Parceiro,False,CLI9914
3,4,2025-08-13,21:45,Casquinha,Cookies,3,30.87,Carvalho de Pereira,PR,Loja Física,True,CLI2071
4,5,2025-04-07,11:45,Picolé,Cookies,4,26.59,Moraes,MA,Loja Física,True,CLI9549


## Nomes Das Colunas E Tipos Identificados

In [5]:
colunas = eda.summarize_columns(df)
colunas

,coluna,tipo_identificado,nulos,percentual_nulos,valores_unicos,exemplo
0,ID_Transacao,int64,0,0.00,50000,1
1,Data,str,0,0.00,213,2025-04-30
2,Hora,str,0,0.00,60,11:00
3,Tipo_Sorvete,str,0,0.00,5,Milkshake
4,Sabor,str,500,1.00,8,Açaí
5,Quantidade,int64,0,0.00,13,3
6,Valor_Total,float64,500,1.00,6786,16.22
7,Cidade,str,500,1.00,6554,Campinas
8,Estado,str,0,0.00,27,AP
9,Canal_Venda,str,0,0.00,3,App


## Valores Nulos

In [6]:
colunas[["coluna", "nulos", "percentual_nulos"]].sort_values(
    ["nulos", "coluna"], ascending=[False, True]
)

,coluna,nulos,percentual_nulos
7,Cidade,500,1.00
4,Sabor,500,1.00
6,Valor_Total,500,1.00
9,Canal_Venda,0,0.00
1,Data,0,0.00
8,Estado,0,0.00
2,Hora,0,0.00
11,ID_Cliente,0,0.00
0,ID_Transacao,0,0.00
10,Promocao,0,0.00


## Duplicidades

In [7]:
eda.check_duplicates(df)

,checagem,quantidade,observacao
0,linhas totalmente duplicadas,0,Duplicidade exata de todos os campos.
1,ID_Transacao duplicado,0,ID_Transacao e indicado como chave primaria no briefing.
2,ID_Cliente distintos,8974,Clientes podem aparecer em multiplas compras.


## Validacao De Formatos

In [8]:
eda.validate_formats(df)

,campo,validacao,registros_invalidos,observacao
0,Data,parse yyyy-mm-dd,0,2025-02-20 a 2025-09-20
1,Hora,parse HH:MM,0,Formato horario valido para todos os registros.
2,Estado,UF com 2 letras maiusculas,0,27 UFs distintas.
3,ID_Cliente,padrao CLI + digitos,0,Formato esperado para identificador de cliente.
4,ID_Transacao,inteiro positivo,0,Chave transacional numerica.


## Inconsistencias Identificadas

In [9]:
eda.identify_inconsistencies(df)

,inconsistencia,quantidade,impacto_potencial
0,Sabor nulo,500,Afeta analises de mix de sabores e ranking de produtos.
1,Valor_Total nulo,500,"Afeta receita, ticket medio e analises financeiras."
2,Cidade nula,500,Afeta leitura geografica e desempenho regional.
3,Quantidade <= 0,1016,"Pode representar devolucao, cancelamento, ajuste ou erro."
4,Valor_Total <= 0,1009,Pode distorcer receita e margem de oportunidade.
5,Valor_Total nulo com Quantidade positiva,493,Indica venda operacional sem valor financeiro registrado.
6,Quantidade positiva e Valor_Total negativo,0,Pode indicar erro de sinal ou regra financeira nao documentada.


## Estatisticas Descritivas

In [10]:
eda.descriptive_statistics(df)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID_Transacao,"50,000.00",NaN,NaN,NaN,"25,000.50","14,433.90",1.00,"12,500.75","25,000.50","37,500.25","50,000.00"
Data,50000,213,2025-05-04,315,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hora,50000,60,19:30,919,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tipo_Sorvete,50000,5,Milkshake,11959,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sabor,49500,8,Açaí,6256,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantidade,"50,000.00",NaN,NaN,NaN,2.98,1.59,-6.00,2.00,3.00,4.00,6.00
Valor_Total,"49,500.00",NaN,NaN,NaN,27.28,17.41,-89.82,13.84,25.23,38.44,89.88
Cidade,49500,6554,Campinas,2461,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Estado,50000,27,AL,1969,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Canal_Venda,50000,3,Parceiro,16771,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Padroes Iniciais

Tabelas simples para entender distribuicoes e concentracoes iniciais. Ainda nao sao KPIs finais, pois dependem de regras de qualidade e tratamento.

In [11]:
padroes = eda.initial_patterns(df)
padroes.keys()

dict_keys(['top_tipos_sorvete_por_registros', 'top_sabores_por_registros', 'canais_por_registros', 'promocao_por_registros', 'receita_por_canal', 'receita_por_tipo_sorvete', 'receita_por_mes'])

In [12]:
padroes["top_tipos_sorvete_por_registros"]

,registros
Tipo_Sorvete,
Milkshake,11959
Sundae,9576
Picolé,9527
Pote,9507
Casquinha,9431


In [13]:
padroes["top_sabores_por_registros"]

,registros
Sabor,
Açaí,6256
Cookies,6253
Caramelo,6232
Morango,6190
Menta,6184
Limão,6181
Baunilha,6164
Chocolate,6040
NaN,500


In [14]:
padroes["canais_por_registros"]

,registros
Canal_Venda,
Parceiro,16771
App,16636
Loja Física,16593


In [15]:
padroes["receita_por_canal"]

,valor_total
Canal_Venda,
Parceiro,"454,528.12"
Loja Física,"448,069.72"
App,"447,947.19"


In [16]:
padroes["receita_por_tipo_sorvete"]

,valor_total
Tipo_Sorvete,
Milkshake,"349,664.25"
Sundae,"252,297.11"
Casquinha,"249,936.42"
Pote,"249,452.59"
Picolé,"249,194.66"


## Campos Provaveis Por Papel Analitico

In [17]:
eda.map_analytical_fields()

,papel_analitico,campos_provaveis
0,data,"Data, Hora"
1,produto,"Tipo_Sorvete, Sabor"
2,canal,Canal_Venda
3,receita,Valor_Total
4,quantidade,Quantidade
5,loja,nao identificado explicitamente
6,regiao,"Cidade, Estado"
7,vendedor,nao identificado explicitamente
8,cliente,ID_Cliente
9,promocao,Promocao


## KPIs Executivos Sugeridos

Sugestoes para uma etapa posterior, apos definicao das regras de tratamento.

In [18]:
eda.suggest_executive_kpis()

,kpi,formula_sugerida,uso_executivo
0,Receita Total,soma de Valor_Total validado,Medir tamanho do negocio e evolucao geral.
1,Ticket Medio,receita valida / numero de transacoes validas,Avaliar monetizacao media por venda.
2,Quantidade Vendida,soma de Quantidade validada,Medir volume e demanda operacional.
3,Receita Por Canal,soma de Valor_Total por Canal_Venda,Priorizar canais com maior retorno.
4,Receita Por Produto/Sabor,soma de Valor_Total por Tipo_Sorvete e Sabor,Identificar mix de produtos mais relevante.
5,Impacto De Promocao,comparar receita e ticket medio entre Promocao=True/False,Avaliar se promocoes geram incremento real.
6,Receita Por Regiao,soma de Valor_Total por Estado/Cidade,Mapear mercados prioritarios.
7,Clientes Recorrentes,clientes com mais de uma transacao / clientes totais,Entender recompra e fidelizacao.


## Possiveis Insights Relacionados A Aumento De Receita

Hipoteses iniciais, nao conclusoes finais. Precisam ser confirmadas apos tratamento dos nulos, valores nao positivos e regras de negocio.

In [19]:
eda.suggest_revenue_insights(df)

,tema,hipotese,proxima_validacao
0,Canal de venda,O canal Parceiro aparece como candidato a alavanca de receita.,"Comparar receita, volume, ticket medio e recorrencia por canal apos tratar dados invalidos."
1,Mix de produtos,Milkshake aparece como categoria candidata a destaque comercial.,"Avaliar receita e quantidade por tipo e sabor, separando promocoes e sazonalidade."
2,Promocoes,"Ticket medio bruto por promocao: {'False': 29.83, 'True': 24.75}.",Medir se promocao aumenta receita incremental ou apenas desloca margem/ticket.
3,Qualidade de dados,Valores nulos e nao positivos podem distorcer a leitura de receita.,"Definir regra de negocio para cancelamentos, devolucoes, erros e vendas sem valor."
4,Geografia,"Cidade e Estado permitem priorizar mercados, mas Cidade possui nulos e alta cardinalidade.",Validar consistencia geografica antes de recomendar expansao regional.


## Conclusoes Da EDA Inicial

- A base possui 50.000 registros e 12 colunas, com estrutura esperada para o case.
- `ID_Transacao` se comporta como chave unica.
- Nao foram encontradas linhas totalmente duplicadas.
- Existem nulos em `Sabor`, `Valor_Total` e `Cidade`.
- Existem valores nao positivos em `Quantidade` e `Valor_Total`, exigindo regra de negocio antes de KPIs financeiros.
- `Data`, `Hora`, `Estado`, `ID_Cliente` e `ID_Transacao` passaram nas validacoes basicas de formato.
- Campos de loja e vendedor nao aparecem explicitamente na base.
- As proximas etapas devem priorizar qualidade dos dados, definicao de regras para valores invalidos e preparacao de uma base tratada em `data/interim` ou `data/processed`.